# Cover Studio（Google Colab）

ポップスの音源から **ピアノカバーの MIDI** を作り、設定を変えながら何度でも作り直して聴き比べられる画面を、Colab の上で動かします。

1. メニューの「ランタイム」→「ランタイムのタイプを変更」で **GPU（T4 など）** を選ぶ
2. 下のセルを実行し、出てきた **「▶ Cover Studio を開く」** をクリックする（新しいタブで開けないときは、その下の「このセルの中で開く」を実行するとノートブックの中に画面が出ます）

あとは画面の中で、音源のアップロード → 採譜（[tsumugi](https://github.com/anime-song/tsumugi)）→ カバーの生成 → 聴き比べ → MIDI / WAV のダウンロードまでできます。

- 初回はライブラリ（PyTorch など）とモデル（tsumugi・カバーモデル）のダウンロードに数分〜10 分ほどかかります
- 画面を使っている間は、このノートブックのタブを開いたままにしてください（閉じたり、しばらく操作しないとランタイムが止まります）
- 採譜は 1 曲あたり数分かかりますが、一度だけです。採譜済みの MIDI を音源と一緒に入れれば採譜を飛ばせます
- カバーモデルの重み（[anime-song/tsumugi-piano-cover](https://huggingface.co/anime-song/tsumugi-piano-cover)）は **CC BY-NC 4.0（非商用）** です

In [ ]:
#@title ▶ Cover Studio を起動する
#@markdown 実行すると、下に「▶ Cover Studio を開く」が出ます。
#@markdown
#@markdown プロジェクト（音源・原曲の MIDI・テイク）を Google ドライブの `MyDrive/cover_studio` に保存して、次回も続きから使う：
USE_DRIVE = True  #@param {type:"boolean"}
REPO_URL = "https://github.com/anime-song/tsumugi-piano-cover"  #@param {type:"string"}
TSUMUGI_URL = "https://github.com/anime-song/tsumugi"  #@param {type:"string"}

import os
import socket
import subprocess
import sys
import time
from pathlib import Path

REPO = Path("/content/tsumugi-piano-cover")
TSUMUGI = Path("/content/tsumugi")
PYTHON = TSUMUGI / ".venv" / "bin" / "python"
PORT = 8100
LOG = Path("/content/cover_studio_server.log")


def running():
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", PORT)) == 0


def run(what, cmd, cwd=None):
    """コマンドを実行する。出力は隠し、失敗したときだけ見せる"""
    print(f"▶ {what}", end="", flush=True)
    start = time.time()
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        print(" … 失敗しました\n" + (r.stdout + r.stderr)[-4000:])
        raise RuntimeError(f"{what}のところで失敗しました（上の出力を見てください）")
    print(f" … 完了（{time.time() - start:.0f} 秒）")


if subprocess.run(["nvidia-smi"], capture_output=True).returncode != 0:
    print("⚠ GPU がありません。「ランタイム」→「ランタイムのタイプを変更」で GPU を選ぶと、採譜と生成がずっと速くなります")

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    DATA = Path("/content/drive/MyDrive/cover_studio")
else:
    DATA = Path("/content/cover_studio_data")
(DATA / "projects").mkdir(parents=True, exist_ok=True)

if not running():
    for url, path in ((REPO_URL, REPO), (TSUMUGI_URL, TSUMUGI)):
        if path.exists():
            run(f"{path.name} を最新にしています", ["git", "-C", str(path), "pull", "-q", "--ff-only"])
        else:
            run(f"{path.name} を取得しています", ["git", "clone", "-q", "--depth", "1", url, str(path)])
    run("uv を入れています", [sys.executable, "-m", "pip", "install", "-q", "uv"])
    run("tsumugi のライブラリをインストールしています（初回は数分かかります）", ["uv", "sync", "--locked", "--extra", "stem"], cwd=TSUMUGI)
    run(
        "Cover Studio のライブラリをインストールしています",
        ["uv", "pip", "install", "-q", "--python", str(PYTHON), "-r", str(REPO / "cover_studio" / "requirements.txt")],
    )
    print("▶ サーバを起動しています（画面とモデルは初回に自動で取ってきます）")
    subprocess.Popen(
        [
            str(PYTHON), "-m", "cover_studio", "--no-browser", "--port", str(PORT),
            "--root", str(DATA / "projects"), "--tsumugi-dir", str(TSUMUGI),
            # カバーモデルは Hugging Face に公開した重みを使う (初めて生成するときにダウンロードする)
            "--pretrained-dir", "", "--checkpoints-dir", "",
        ],
        cwd=REPO, stdout=open(LOG, "a"), stderr=subprocess.STDOUT, start_new_session=True,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    for _ in range(180):
        if running():
            break
        time.sleep(1)
    else:
        raise RuntimeError(f"サーバが起動しませんでした。下の「サーバのログを見る」か {LOG} を確認してください")

from google.colab import output

print("✅ 準備できました。下のリンクから開いてください（開けないときは、下の「このセルの中で開く」を実行してください）")
output.serve_kernel_port_as_window(PORT, anchor_text="▶ Cover Studio を開く")

In [ ]:
#@title （開けないとき）このセルの中で開く
#@markdown 「▶ Cover Studio を開く」で新しいタブが開かない・真っ白になる・エラーになるときは、このセルを実行するとノートブックの中に画面を出します。
#@markdown 先に上の「Cover Studio を起動する」を実行しておいてください。
#@markdown
#@markdown 画面の高さ（ピクセル）：
HEIGHT = 900  #@param {type:"integer"}

import socket

from google.colab import output

PORT = 8100

with socket.socket() as s:
    if s.connect_ex(("127.0.0.1", PORT)) != 0:
        raise RuntimeError("サーバが動いていません。先に上の「Cover Studio を起動する」を実行してください")

print("MIDI のダウンロードがうまくいかないときは、Google ドライブの MyDrive/cover_studio/projects/<曲>/takes/<テイク>/cover.mid にも保存されています（USE_DRIVE が有効なとき）")
output.serve_kernel_port_as_iframe(PORT, height=HEIGHT)

In [ ]:
#@title （困ったとき）サーバのログを見る
!tail -n 100 /content/cover_studio_server.log

In [ ]:
#@title （困ったとき）サーバを止める
#@markdown 止めたあと、上の「Cover Studio を起動する」をもう一度実行すると起動し直します。
import subprocess
import time

subprocess.run(["pkill", "-f", "cover_studio --no-browser"])
time.sleep(2)
print("サーバを止めました。上のセルをもう一度実行すると起動し直します")